# Semantic Saturation AnalysisWhen does the model "know" its answer? Track how semantic information accumulates across layers and identify redundant computation.Key insight: In many cases, the model's prediction is largely determined by early or middle layers, with later layers contributing diminishing returns.

## Why This Matters

Semantic saturation analysis identifies where in the network new information stops being added. Information saturation, redundant layers, and stabilization points reveal which layers are doing useful work and which are largely passing information through — informing both pruning and circuit analysis.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformerfrom irtk.semantic_saturation import (    semantic_information_saturation,    redundant_layer_detection,    token_saturation_curve,    representation_stabilization_point,    early_vs_late_computation_balance,)cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

## Semantic Information SaturationTrack the probability of a target token across layers using logit lens. The "saturation layer" is where the model first achieves 90% of its final confidence.

In [ ]:
result = semantic_information_saturation(model, tokens, target_token=5)print(f"Final probability for token 5: {result['final_prob']:.6f}")print(f"Saturation layer: {result['saturation_layer']}")for l, p in enumerate(result['layer_probs']):    gain = result['information_gain'][l]    print(f"  Layer {l}: prob={p:.6f}, gain={gain:+.6f}")

## Redundant Layer DetectionWhich layers can be removed with minimal impact? Zero out each layer's contribution independently and measure the effect.

In [ ]:
result = redundant_layer_detection(model, tokens, metric)print(f"Layer effects: {[f'{e:.6f}' for e in result['layer_effects']]}")print(f"Redundant layers: {result['redundant_layers']}")print(f"Essential layers: {result['essential_layers']}")print(f"Most redundant: layer {result['most_redundant']}")

## Token Saturation CurveEntropy of the output distribution at each layer — how quickly does the model narrow down its prediction from uniform to confident?

In [ ]:
result = token_saturation_curve(model, tokens)for l, e in enumerate(result['layer_entropies']):    print(f"  Layer {l}: entropy = {e:.4f}")print(f"Total entropy reduction: {result['entropy_reduction']:.4f}")print(f"Steepest drop at layer: {result['steepest_drop_layer']}")

## Representation StabilizationAt what layer do the residual stream representations stop changing significantly?

In [ ]:
result = representation_stabilization_point(model, tokens)for i, d in enumerate(result['layer_distances']):    print(f"  Layers {i}->{i+1}: cosine distance = {d:.6f}")print(f"Total drift: {result['total_drift']:.6f}")print(f"Stabilization layer: {result['stabilization_layer']}")

## Early vs Late Computation BalanceIs the model doing most of its work in early layers or late layers?

In [ ]:
result = early_vs_late_computation_balance(model, tokens, metric)print(f"Early effect: {result['early_effect']:.6f}")print(f"Late effect: {result['late_effect']:.6f}")print(f"Balance ratio: {result['balance_ratio']:.4f}")print(f"  (0 = all late, 0.5 = balanced, 1 = all early)")print(f"Split at layer: {result['split_layer']}")